In [ ]:
import importlib
import NeuralNetwork

importlib.reload(NeuralNetwork)
importlib.reload(funcs)
from NeuralNetwork import NeuralNetwork
import funcs

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from collections import defaultdict
import numpy as np


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device being used: {device.type}")

## Global parameters

In [ ]:
# model parameters
HIDDEN_LAYERS = [512, 256, 128]
TRAIN_VAL_SPLIT = 0.8

# initial training
N_TRAIN_EPOCHS = 15 if device.type == "cuda" else 8

# pruning loop
MAX_ALLOWED_ACC_DROP = 0.01
MAX_PRUNE_ROUNDS = 10
PRUNE_FRAC = 0.2
REGROW_FRAC = 0.1
MIN_VAL_ACC = 0.9
N_RETRIAN_EPOCHS = 3

# clustering
N_CLUSTERS = 8

## Setting up data and initial model

In [ ]:
# Define a transform to convert images to PyTorch tensors
transform = transforms.ToTensor()

# Download and load the training set
train_dataset = datasets.EMNIST(
    root="./data",       # where to store the data
    split="digits",      # "digits" for 0–9, other options: "letters", "balanced", etc.
    train=True,
    download=True,
    transform=transform
)

# Download and load the test set
test_dataset = datasets.EMNIST(
    root="./data",
    split="digits",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
train_size = int(TRAIN_VAL_SPLIT * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])
print(f"train size: {train_size}, val size: {val_size}, test size: {len(test_dataset)}")

In [ ]:
# Create a DataLoader for batching
batch_size = 256 if device.type == "cuda" else 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
# Create model and train
model = NeuralNetwork(hidden_sizes=HIDDEN_LAYERS, device=device)

In [ ]:
model.train_model(train_loader=train_loader, val_loader=val_loader, epochs=N_TRAIN_EPOCHS)

In [ ]:
train_acc = model.accuracy(train_loader)
val_acc = model.accuracy(val_loader)

In [ ]:
print(f"Train: {(train_acc*100):.2f}%, Validation: {(val_acc*100):.2f}%")

## Prune Neurons and Retrain

In [ ]:
prune_parameters = (MAX_PRUNE_ROUNDS, PRUNE_FRAC, REGROW_FRAC, N_RETRIAN_EPOCHS, MAX_ALLOWED_ACC_DROP)
use_max_rounds = False if device.type == "cuda" else True
final_model, final_val_acc, metrics_history = funcs.pruning(model, train_loader, val_loader, prune_parameters, use_max_rounds)

In [ ]:
for round_idx, metrics in enumerate(metrics_history, start=1):
    print(f"--- Round {round_idx} ---")
    final_val_acc = metrics['val_acc'].iloc[-1]
    print(f"Final validation accuracy: {final_val_acc:.5f}")

In [ ]:
print(f"Test accuracy after pruning: {final_model.accuracy(test_loader)*100:.2f}")

## Activation analysis and Neuron Clustering

In [ ]:
layer_data_post_pruning = final_model.get_layer_data(train_loader)

In [ ]:
for key, value in layer_data_post_pruning.items():
    print(f"{key} has shape: {value['post_activation'].shape}")

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster

hidden_layers = [k for k in layer_data_post_pruning.keys() if 'layer_' in k]

all_neuron_activations = []
for layer_name in hidden_layers:
    acts = layer_data_post_pruning[layer_name]['post_activation']
    all_neuron_activations.append(acts)

all_neuron_activations = torch.cat(all_neuron_activations, dim=1)

layer_mapping = []
start_idx = 0
for layer_name in hidden_layers:
    n_neurons = layer_data_post_pruning[layer_name]['post_activation'].shape[1]
    end_idx = start_idx + n_neurons
    layer_mapping.append((layer_name, start_idx, end_idx))
    start_idx

neurons_for_clustering = all_neuron_activations.T.numpy()

Z = linkage(neurons_for_clustering, method='ward', metric='euclidean')

n_clusters = N_CLUSTERS
clusters = fcluster(Z, t=n_clusters, criterion='maxclust')

print(f"Total neurons clustered: {neurons_for_clustering.shape[0]}")
print(f"Clusters assigned: {np.unique(clusters)}")

In [ ]:
cluster_map = defaultdict(list)
for neuron_idx, cluster_id in enumerate(clusters):
    cluster_map[cluster_id].append(neuron_idx)

cluster_results = {}
for cluster_id, neuron_indices in cluster_map.items():
    per_class_acc = funcs.cluster_criticality_per_class(
        final_model,
        neuron_indices,
        layer_mapping,
        val_loader,
        cluster_id,
        device=device
    )
    cluster_results[cluster_id] = per_class_acc

In [ ]:
cluster_class_changes = funcs.plot_cluster_accuracy_bars(cluster_results, target_labels=list(range(10)))

## cluster extraction

In [ ]:
cluster_assignment = {} # per layer cluster labels